# AutoGrow4 Results Viewer

Standalone notebook for viewing results from any AutoGrow4 run.
No need to run the main AutoGrow4 notebook first.

In [1]:
# Fix nglview frontend version mismatch on SDSC Expanse
# Must patch the source file BEFORE importing nglview, since traitlets
# are set at class definition time and cannot be overridden at runtime.
import warnings, os, re, subprocess, sys
warnings.filterwarnings("ignore", message=".*pkg_resources.*")

# Step 1: Detect what JupyterLab has installed
_lab_version = None
try:
    _r = subprocess.run(['jupyter', 'labextension', 'list'],
                       capture_output=True, text=True, timeout=10)
    for line in (_r.stdout + _r.stderr).split('\n'):
        if 'nglview-js-widgets' in line:
            m = re.search(r'v(\d+\.\d+\.\d+)', line)
            if m:
                _lab_version = m.group(1)
                break
except Exception:
    pass

if _lab_version:
    print(f'JupyterLab nglview-js-widgets version: {_lab_version}')
    
    # Step 2: Find and patch nglview _frontend.py on disk BEFORE importing
    for path in sys.path:
        frontend_file = os.path.join(path, 'nglview', '_frontend.py')
        if os.path.exists(frontend_file):
            with open(frontend_file) as f:
                content = f.read()
            m = re.search(r"__frontend_version__\s*=\s*'([^']+)'", content)
            if m and m.group(1) != _lab_version:
                old_ver = m.group(1)
                new_content = content.replace(
                    f"__frontend_version__ = '{old_ver}'",
                    f"__frontend_version__ = '{_lab_version}'"
                )
                with open(frontend_file, 'w') as f:
                    f.write(new_content)
                print(f'Patched {frontend_file}: {old_ver} -> {_lab_version}')
            elif m:
                print(f'nglview _frontend.py already matches ({m.group(1)})')
            break

# Step 3: Now import nglview (will use the patched version)
try:
    import nglview
    print(f'nglview loaded successfully (version {nglview.__version__})')
except Exception as e:
    print(f'nglview import issue: {e}')
    print('Scoring and results viewing will still work.')

JupyterLab nglview-js-widgets version: 3.1.5
nglview _frontend.py already matches (3.1.5)


nglview loaded successfully (version 0.0.0)


---
## 1. Select Run & View Top Scores

In [2]:
import glob
import os
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

PROJECT_DIR = os.path.expanduser("~/final_project")

def _find_output_dirs():
    """Find all directories containing AutoGrow4 Run_* output folders."""
    output_dirs = []
    # Search in project directory and all subdirectories
    for candidate in sorted(glob.glob(os.path.join(PROJECT_DIR, "*"))):
        if os.path.isdir(candidate):
            runs = glob.glob(os.path.join(candidate, "Run_*"))
            if runs:
                output_dirs.append(candidate)
    if glob.glob(os.path.join(PROJECT_DIR, "Run_*")):
        output_dirs.append(PROJECT_DIR)
    # Also search Lustre scratch storage
    lustre_scratch = os.path.expanduser("~/lustre_scratch")
    if os.path.isdir(lustre_scratch):
        for candidate in sorted(glob.glob(os.path.join(lustre_scratch, "*"))):
            if os.path.isdir(candidate):
                runs = glob.glob(os.path.join(candidate, "Run_*"))
                if runs:
                    output_dirs.append(candidate)
        if glob.glob(os.path.join(lustre_scratch, "Run_*")):
            output_dirs.append(lustre_scratch)
    return sorted(set(output_dirs))

def _find_runs(output_dir):
    return sorted(glob.glob(os.path.join(output_dir, "Run_*")))

def _find_generations(run_dir):
    return sorted(glob.glob(os.path.join(run_dir, "generation_*", "*ranked*.smi")))

def load_ranked_smi(filepath):
    """Load a ranked .smi file into a DataFrame."""
    rows = []
    with open(filepath) as f:
        for line in f:
            parts = line.strip().split("\t")
            if len(parts) >= 2:
                row = {"SMILES": parts[0], "Name": parts[1]}
                if len(parts) >= 3:
                    try: row["Docking_Score"] = float(parts[2])
                    except ValueError: pass
                if len(parts) >= 4:
                    try: row["Diversity_Score"] = float(parts[3])
                    except ValueError: pass
                rows.append(row)
    return pd.DataFrame(rows)

# Build UI
_output_dirs = _find_output_dirs()

w_results_output_dir = widgets.Dropdown(
    options=[(os.path.basename(d), d) for d in _output_dirs] if _output_dirs else [("No output directories found", "")],
    description="Output Dir:",
    style={"description_width": "100px"},
    layout=widgets.Layout(width="500px")
)
w_results_run = widgets.Dropdown(options=[], description="Run:",
    style={"description_width": "100px"}, layout=widgets.Layout(width="500px"))
w_results_gen = widgets.Dropdown(options=[], description="Generation:",
    style={"description_width": "100px"}, layout=widgets.Layout(width="500px"))
_refresh_btn = widgets.Button(description="Refresh", button_style="info", layout=widgets.Layout(width="100px"))
_show_btn = widgets.Button(description="Show Results", button_style="success", layout=widgets.Layout(width="150px"))
results_output = widgets.Output()

def _update_runs(change=None):
    d = w_results_output_dir.value
    if d:
        runs = _find_runs(d)
        w_results_run.options = [(os.path.basename(r), r) for r in runs] if runs else [("No runs", "")]
        if runs: _update_gens()

def _update_gens(change=None):
    r = w_results_run.value
    if r:
        ranked = _find_generations(r)
        w_results_gen.options = [(f"{os.path.basename(os.path.dirname(f))} ({os.path.getsize(f)} bytes)", f)
                                  for f in ranked] if ranked else [("No ranked files", "")]

def _refresh(b=None):
    dirs = _find_output_dirs()
    w_results_output_dir.options = [(os.path.basename(d), d) for d in dirs] if dirs else [("None found", "")]
    if dirs: _update_runs()

def _show(b):
    results_output.clear_output()
    with results_output:
        f = w_results_gen.value
        if not f or not os.path.exists(f):
            print("No ranked file selected.")
            return
        gen = os.path.basename(os.path.dirname(f))
        run = os.path.basename(os.path.dirname(os.path.dirname(f)))
        display(HTML(f"<h3>Results: {run} / {gen}</h3>"))
        df = load_ranked_smi(f)
        if df.empty:
            print("Ranked file is empty (0 bytes). All dockings may have failed.")
            return
        display(HTML(f"<p><b>{len(df)} ranked molecules</b></p>"))
        if "Docking_Score" in df.columns:
            display(HTML(f"<p>Best: {df['Docking_Score'].min():.3f} kcal/mol | "
                         f"Average: {df['Docking_Score'].mean():.3f} kcal/mol</p>"))
        display(HTML("<p><b>Top 20 molecules:</b></p>"))
        display(df.head(20))
        # All generations summary
        run_dir = w_results_run.value
        all_ranked = _find_generations(run_dir)
        if len(all_ranked) > 1:
            display(HTML("<hr><h4>All Generations Summary</h4>"))
            rows = []
            for rf in all_ranked:
                gn = os.path.basename(os.path.dirname(rf))
                gdf = load_ranked_smi(rf)
                if not gdf.empty and "Docking_Score" in gdf.columns:
                    rows.append({"Generation": gn, "Molecules": len(gdf),
                                 "Best": gdf["Docking_Score"].min(),
                                 "Average": gdf["Docking_Score"].mean(),
                                 "Worst": gdf["Docking_Score"].max()})
                else:
                    rows.append({"Generation": gn, "Molecules": len(gdf),
                                 "Best": None, "Average": None, "Worst": None})
            if rows:
                display(pd.DataFrame(rows))

w_results_output_dir.observe(_update_runs, names="value")
w_results_run.observe(_update_gens, names="value")
_refresh_btn.on_click(_refresh)
_show_btn.on_click(_show)
_update_runs()

display(widgets.VBox([
    widgets.HTML("<p>Select output directory, run, and generation. Click Refresh to rescan.</p>"),
    widgets.HBox([w_results_output_dir, _refresh_btn]),
    w_results_run,
    w_results_gen,
    _show_btn,
    results_output
]))

---
## 2. Fitness Progression Plot

In [4]:
import matplotlib.pyplot as plt
import numpy as np

plot_output = widgets.Output()
_plot_btn = widgets.Button(description="Plot Fitness", button_style="primary", layout=widgets.Layout(width="150px"))

def _plot(b):
    plot_output.clear_output()
    with plot_output:
        run_dir = w_results_run.value
        if not run_dir:
            print("Select a Run first above.")
            return
        ranked_files = sorted(glob.glob(os.path.join(run_dir, "generation_*", "*ranked*.smi")))
        if not ranked_files:
            print("No ranked files found.")
            return
        gen_numbers, best_scores, avg_scores = [], [], []
        for rf in ranked_files:
            gen_num = int(os.path.basename(os.path.dirname(rf)).replace("generation_", ""))
            scores = []
            with open(rf) as f:
                for line in f:
                    parts = line.strip().split("\t")
                    if len(parts) >= 3:
                        try: scores.append(float(parts[2]))
                        except ValueError: pass
            if scores:
                gen_numbers.append(gen_num)
                best_scores.append(min(scores))
                avg_scores.append(np.mean(scores))
        if gen_numbers:
            fig, ax = plt.subplots(figsize=(10, 5))
            ax.plot(gen_numbers, best_scores, "o-", label="Best Score", color="#e74c3c", linewidth=2)
            ax.plot(gen_numbers, avg_scores, "s--", label="Average Score", color="#3498db", linewidth=1.5)
            ax.set_xlabel("Generation", fontsize=12)
            ax.set_ylabel("Docking Score (kcal/mol)", fontsize=12)
            ax.set_title("AutoGrow4 Fitness Progression", fontsize=14)
            ax.legend(); ax.grid(True, alpha=0.3)
            plt.tight_layout(); plt.show()
        else:
            print("No valid scores found.")

_plot_btn.on_click(_plot)
display(widgets.VBox([_plot_btn, plot_output]))

---
## 2b. Molecular Weight vs Docking Score Analysis

Compare docking scores with molecular weights across generations.  
Higher MW molecules may score well but be less drug-like. Use this to find the
best score-to-weight tradeoff and decide which generation to proceed with.

In [ ]:
from rdkit import Chem
from rdkit.Chem import Descriptors
import matplotlib.pyplot as plt
import numpy as np

mw_output = widgets.Output()

w_mw_top_n = widgets.IntSlider(value=20, min=5, max=100, step=5,
    description='Top N:', style={'description_width': '60px'},
    layout=widgets.Layout(width='300px'))
_mw_btn = widgets.Button(description='Analyze MW vs Score',
    button_style='primary', layout=widgets.Layout(width='200px'))

def _calc_mw(smiles):
    """Calculate molecular weight from SMILES string."""
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol:
            return round(Descriptors.MolWt(mol), 2)
    except Exception:
        pass
    return None

def _analyze_mw(b):
    mw_output.clear_output()
    with mw_output:
        run_dir = w_results_run.value
        if not run_dir:
            print('Select a Run first in Section 1 above.')
            return
        ranked_files = sorted(glob.glob(os.path.join(run_dir, 'generation_*', '*ranked*.smi')))
        if not ranked_files:
            print('No ranked files found.')
            return

        top_n = w_mw_top_n.value
        print(f'Analyzing top {top_n} molecules per generation...\n')

        # Collect data across all generations
        gen_data = []
        for rf in ranked_files:
            gen_name = os.path.basename(os.path.dirname(rf))
            gen_num = int(gen_name.replace('generation_', ''))
            df = load_ranked_smi(rf)
            if df.empty or 'Docking_Score' not in df.columns:
                continue
            df = df.head(top_n).copy()
            df['MW'] = df['SMILES'].apply(_calc_mw)
            df = df.dropna(subset=['MW'])
            if df.empty:
                continue
            gen_data.append({
                'gen_num': gen_num,
                'gen_name': gen_name,
                'df': df,
                'avg_score': df['Docking_Score'].mean(),
                'best_score': df['Docking_Score'].min(),
                'avg_mw': df['MW'].mean(),
                'min_mw': df['MW'].min(),
                'max_mw': df['MW'].max(),
                'best_ligand': df.iloc[0]['Name'] if len(df) > 0 else '',
                'best_mw': df.iloc[0]['MW'] if len(df) > 0 else 0,
                'best_dock': df.iloc[0]['Docking_Score'] if len(df) > 0 else 0,
                'score_std': df['Docking_Score'].std(),
                'mw_std': df['MW'].std(),
            })

        if not gen_data:
            print('No valid data found.')
            return

        # Summary table
        summary = pd.DataFrame([{
            'Generation': g['gen_name'],
            'Avg Score': round(g['avg_score'], 2),
            'Score StdDev': round(g['score_std'], 2),
            'Best Score': round(g['best_score'], 2),
            'Avg MW': round(g['avg_mw'], 1),
            'MW StdDev': round(g['mw_std'], 1),
            'MW Range': f"{g['min_mw']:.0f}-{g['max_mw']:.0f}",
            'Score/MW': round(g['avg_score'] / g['avg_mw'] * 100, 3),
            'Best Ligand': g['best_ligand'],
            'Best MW': g['best_mw'],
        } for g in gen_data])
        display(HTML(f'<h4>Top {top_n} Molecules: Score vs Molecular Weight</h4>'))
        display(summary)
        print(f'\nScore/MW ratio = (avg_score / avg_MW) * 100')
        print('Lower Score/MW ratio = better binding efficiency per unit weight')

        # Plot 1: Avg Score and Avg MW across generations (dual axis)
        fig, ax1 = plt.subplots(figsize=(10, 5))
        gens = [g['gen_num'] for g in gen_data]
        scores = [g['avg_score'] for g in gen_data]
        mws = [g['avg_mw'] for g in gen_data]

        color1 = '#e74c3c'
        ax1.set_xlabel('Generation', fontsize=12)
        ax1.set_ylabel('Avg Docking Score (kcal/mol)', color=color1, fontsize=12)
        ax1.plot(gens, scores, 'o-', color=color1, linewidth=2, label='Avg Score')
        ax1.tick_params(axis='y', labelcolor=color1)

        ax2 = ax1.twinx()
        color2 = '#2ecc71'
        ax2.set_ylabel('Avg Molecular Weight (Da)', color=color2, fontsize=12)
        ax2.plot(gens, mws, 's--', color=color2, linewidth=2, label='Avg MW')
        ax2.tick_params(axis='y', labelcolor=color2)

        ax1.set_title(f'Docking Score vs Molecular Weight (Top {top_n})', fontsize=14)
        lines1, labels1 = ax1.get_legend_handles_labels()
        lines2, labels2 = ax2.get_legend_handles_labels()
        ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')
        ax1.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

        # Plot 2: Scatter of Score vs MW for each generation
        fig, ax = plt.subplots(figsize=(10, 6))
        cmap = plt.cm.viridis
        for i, g in enumerate(gen_data):
            color = cmap(i / max(len(gen_data) - 1, 1))
            ax.scatter(g['df']['MW'], g['df']['Docking_Score'],
                      color=color, alpha=0.6, s=40, label=f"Gen {g['gen_num']}")
        ax.set_xlabel('Molecular Weight (Da)', fontsize=12)
        ax.set_ylabel('Docking Score (kcal/mol)', fontsize=12)
        ax.set_title(f'Score vs MW Scatter (Top {top_n} per generation)', fontsize=14)
        ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

        # Detailed per-generation top 5
        display(HTML('<hr><h4>Top 5 Ligands Per Generation (with MW)</h4>'))
        for g in gen_data:
            top5 = g['df'].head(5)[['Name', 'SMILES', 'Docking_Score', 'MW']].copy()
            top5.columns = ['Name', 'SMILES', 'Score', 'MW (Da)']
            display(HTML(f'<b>{g["gen_name"]}</b>'))
            display(top5)

_mw_btn.on_click(_analyze_mw)
display(widgets.VBox([w_mw_top_n, _mw_btn, mw_output]))

---
## 3. Docking Visualization

In [3]:
import nglview as nv
import time

viz_output = widgets.Output()
w_ligand_select = widgets.Dropdown(options=[], description="Ligand:",
    style={"description_width": "80px"}, layout=widgets.Layout(width="600px"))
w_receptor_path = widgets.Text(
    value=os.path.expanduser("~/final_project/receptors/2ZSH.pdbqt"),
    description="Receptor:", style={"description_width": "80px"},
    layout=widgets.Layout(width="600px"))
w_binding_residues = widgets.Text(
    value="115, 116, 191, 289, 319",
    description="Key residues:",
    style={"description_width": "100px"},
    layout=widgets.Layout(width="600px"),
    tooltip="Comma-separated residue numbers (PDB numbering) to show as sticks")
_load_btn = widgets.Button(description="Load Ligands", button_style="info", layout=widgets.Layout(width="150px"))
_show_dock_btn = widgets.Button(description="Show Docking", button_style="success", layout=widgets.Layout(width="150px"))

_mapping_note = widgets.HTML(
    value="<small><i>2ZSH PDB numbering offset: subtract 7 from literature/UniProt numbers. "
          "e.g. Gly122&#8594;115, Ser123&#8594;116, Ser198&#8594;191, Asp296&#8594;289, Val326&#8594;319</i></small>")

def _load_ligands(b):
    viz_output.clear_output()
    with viz_output:
        run_dir = w_results_run.value
        if not run_dir:
            print("Select a Run first above.")
            return
        gen_dirs = sorted(glob.glob(os.path.join(run_dir, "generation_*")))
        if not gen_dirs:
            print("No generation directories found.")
            return
        latest_gen = gen_dirs[-1]
        pdbs_dir = os.path.join(latest_gen, "PDBs")

        # Check for compressed files
        compressed = os.path.join(pdbs_dir, "compressed_PDBS.txt.gz")
        if os.path.exists(compressed) and not glob.glob(os.path.join(pdbs_dir, "*.pdb")):
            print("PDB files are compressed. Decompressing...")
            try:
                import gzip
                with gzip.open(compressed, "rt") as gz:
                    content = gz.read()
                current_file = None
                current_lines = []
                for line in content.split("\n"):
                    if line.startswith(">>>>"):
                        if current_file and current_lines:
                            with open(os.path.join(pdbs_dir, current_file), "w") as wf:
                                wf.write("\n".join(current_lines))
                        current_file = line[4:].strip()
                        current_lines = []
                    else:
                        current_lines.append(line)
                if current_file and current_lines:
                    with open(os.path.join(pdbs_dir, current_file), "w") as wf:
                        wf.write("\n".join(current_lines))
                print("Done.")
            except Exception as e:
                print(f"Decompression error: {e}")
                return

        # Prefer .vina files (docked poses with correct coordinates)
        # .pdb/.pdbqt files are pre-docking conformations (wrong coordinates!)
        vina = sorted(glob.glob(os.path.join(pdbs_dir, "*.vina")))
        pdbqt = sorted(glob.glob(os.path.join(pdbs_dir, "*.pdbqt")))
        pdb = sorted(glob.glob(os.path.join(pdbs_dir, "*.pdb")))

        if vina:
            # Show .vina files first - these have the docked coordinates
            w_ligand_select.options = [("[DOCKED] " + os.path.basename(f), f) for f in vina]
            print(f"Found {len(vina)} docked poses (.vina) in {os.path.basename(latest_gen)}")
            print("These contain the Vina docked coordinates aligned with the receptor.")
        elif pdbqt or pdb:
            all_files = pdbqt + pdb
            w_ligand_select.options = [(os.path.basename(f), f) for f in all_files]
            print(f"Found {len(pdbqt)} PDBQT, {len(pdb)} PDB files (pre-docking conformations)")
            print("WARNING: No .vina files found. These are pre-docking coordinates and may not align with the receptor.")
        else:
            print(f"No ligand files in {os.path.basename(latest_gen)}/PDBs/")
            print("Files may have been cleaned up (reduce_files was enabled).")

def _pdbqt_to_pdb(text):
    """Convert PDBQT text to valid PDB by stripping extra columns and PDBQT keywords."""
    pdb_lines = []
    for line in text.split('\n'):
        record = line[:6].strip()
        if record in ('ROOT', 'ENDROOT', 'BRANCH', 'ENDBRANCH', 'TORSDOF'):
            continue
        if record in ('ATOM', 'HETATM'):
            pdb_lines.append(line[:66].rstrip())
        elif record in ('TER', 'END', 'MODEL', 'ENDMDL', 'REMARK', 'HEADER', 'TITLE', 'CONECT'):
            pdb_lines.append(line.rstrip())
    return '\n'.join(pdb_lines)

def _extract_best_pose(text):
    """Extract only MODEL 1 (best pose) from a Vina output file."""
    lines = text.split('\n')
    model_lines = []
    in_model_1 = False
    for line in lines:
        stripped = line.strip()
        if stripped == 'MODEL 1':
            in_model_1 = True
            continue
        elif stripped.startswith('MODEL ') and stripped != 'MODEL 1':
            break  # Stop at MODEL 2+
        elif stripped == 'ENDMDL':
            if in_model_1:
                break
            continue
        if in_model_1:
            model_lines.append(line)
    # If no MODEL records, use the whole file
    if not model_lines:
        model_lines = lines
    return '\n'.join(model_lines)

def _read_structure(filepath):
    """Read a structure file and return (text, ext) for nglview.
    
    For .vina files: extracts best docked pose (MODEL 1) and converts PDBQT->PDB.
    For .pdbqt files: converts to PDB format.
    For .pdb files: uses as-is.
    """
    with open(filepath) as f:
        text = f.read()
    ext = os.path.splitext(filepath)[1].lstrip(".")
    
    if ext == "vina":
        # Vina output: extract best pose, then convert PDBQT->PDB
        text = _extract_best_pose(text)
        text = _pdbqt_to_pdb(text)
        ext = "pdb"
    elif ext == "pdbqt":
        text = _pdbqt_to_pdb(text)
        ext = "pdb"
    return text, ext

def _show_docking(b):
    viz_output.clear_output()
    with viz_output:
        lig = w_ligand_select.value
        rec = w_receptor_path.value
        if not lig:
            print("Select a ligand first.")
            return
        if not os.path.exists(lig):
            print(f"Ligand not found: {lig}")
            return

        # If user selected a .pdb or .pdbqt, try to find the .vina version
        lig_ext = os.path.splitext(lig)[1]
        if lig_ext in ('.pdb', '.pdbqt'):
            base = lig.rsplit('.', 1)[0]
            vina_path = base + '.pdbqt.vina'
            if not os.path.exists(vina_path):
                vina_path = base + '.vina'
            if os.path.exists(vina_path):
                print(f"Using docked pose: {os.path.basename(vina_path)}")
                lig = vina_path
            else:
                print(f"WARNING: No .vina docked pose found. Using pre-docking coordinates.")
                print(f"  Ligand may appear far from receptor.")

        print(f"Loading: {os.path.basename(lig)}")
        try:
            lig_text, lig_ext = _read_structure(lig)
            has_rec = os.path.exists(rec) if rec else False

            lig_atoms = sum(1 for l in lig_text.split('\n') if l[:6].strip() in ('ATOM', 'HETATM'))
            print(f"Ligand atoms: {lig_atoms}")

            if has_rec:
                rec_text, rec_ext = _read_structure(rec)
                rec_atoms = sum(1 for l in rec_text.split('\n') if l[:6].strip() in ('ATOM', 'HETATM'))
                print(f"Receptor atoms: {rec_atoms}")

                view = nv.NGLWidget()
                view._remote_call("setSize", target="Widget", args=["100%", "500px"])

                # Add receptor as component 0
                view.add_component(nv.TextStructure(rec_text, ext=rec_ext),
                                   default_representation=False, name="receptor")
                time.sleep(0.3)
                view.add_cartoon(selection="protein", color="sstruc", opacity=0.8, component=0)

                # Show key binding site residues
                residue_text = w_binding_residues.value.strip()
                if residue_text:
                    res_nums = [r.strip() for r in residue_text.split(",") if r.strip()]
                    if res_nums:
                        sel = "(" + " or ".join(res_nums) + ") and sidechainAttached"
                        view.add_licorice(selection=sel, color="element", component=0)
                        label_sel = "(" + " or ".join(res_nums) + ") and .CA"
                        view.add_label(selection=label_sel, color="white",
                                       labelType="format", labelFormat="%(resname)s%(resno)s",
                                       labelGrouping="residue", component=0,
                                       zOffset=2.0, fixedSize=True,
                                       attachment="middle_center", showBackground=True,
                                       backgroundColor="black", backgroundOpacity=0.6)
                        print(f"Showing side chains: {', '.join(res_nums)}")

                # Add ligand as component 1
                view.add_component(nv.TextStructure(lig_text, ext=lig_ext),
                                   default_representation=False, name="ligand")
                time.sleep(0.3)
                view.add_ball_and_stick(selection="all", color="element",
                                        aspectRatio=2.0, component=1)
                view.add_label(selection="all", color="yellow",
                               labelType="format", labelFormat="LIGAND",
                               labelGrouping="structure", component=1,
                               zOffset=3.0, fixedSize=True,
                               attachment="middle_center", showBackground=True,
                               backgroundColor="green", backgroundOpacity=0.7)
            else:
                print("Receptor not found, showing ligand only.")
                view = nv.NGLWidget()
                view.add_component(nv.TextStructure(lig_text, ext=lig_ext),
                                   default_representation=False, name="ligand")
                time.sleep(0.3)
                view.add_ball_and_stick(selection="all", color="element", component=0)

            view.center()
            display(view)
            print("Tip: rotate to see the ligand in the active site. Scroll to zoom.")
        except Exception as e:
            print(f"Visualization error: {e}")
            import traceback
            traceback.print_exc()

_load_btn.on_click(_load_ligands)
_show_dock_btn.on_click(_show_docking)

display(widgets.VBox([
    w_receptor_path,
    w_binding_residues,
    _mapping_note,
    widgets.HBox([_load_btn, _show_dock_btn]),
    w_ligand_select,
    viz_output
]))

---
## 4. Export Top Compounds to CSV

In [ ]:
export_output = widgets.Output()
_export_btn = widgets.Button(description="Export Top 50 to CSV", button_style="warning", layout=widgets.Layout(width="200px"))

def _export(b):
    export_output.clear_output()
    with export_output:
        f = w_results_gen.value
        if not f or not os.path.exists(f):
            print("Select a generation first.")
            return
        df = load_ranked_smi(f)
        if df.empty:
            print("No data to export.")
            return
        run_name = os.path.basename(os.path.dirname(os.path.dirname(f)))
        gen_name = os.path.basename(os.path.dirname(f))
        csv_path = os.path.join(PROJECT_DIR, f"top_compounds_{run_name}_{gen_name}.csv")
        df.head(50).to_csv(csv_path, index=False)
        print(f"Exported top 50 compounds to: {csv_path}")
        display(df.head(50))

_export_btn.on_click(_export)
display(widgets.VBox([_export_btn, export_output]))